## Libraries

In [17]:
import requests
import pandas as pd
from tqdm import tqdm

In [50]:
# Step 1: Get the list of publications for the institution
institution_url = "https://www.croris.hr/crosbi-api/ustanova/289"

# Fetch the list of publications
response = requests.get(institution_url)
if response.status_code == 200:
    data = response.json()
    publications = data['_links']['publikacije']  # Get the list of publications
else:
    print(f"Error fetching data from {institution_url}. Status code: {response.status_code}")

def authorstringlist2dict(stringlist):
    authorlist = []
    if ";" in stringlist:
        for author in stringlist.split(";"):
            authorlist.append({"name": author.split(",")[1], "surname": author.split(",")[0], "authorId": author.replace(",", "").replace(" ", ""), "role": "autor/i"})
    else:
        for author in [stringlist]:
            authorlist.append({"name": author.split(",")[1], "surname": author.split(",")[0], "authorId": author.replace(",", "").replace(" ", ""), "role": "autor/i"})
        
    return authorlist

data_list = []
# Step 2: Loop through each publication and fetch details
for publication in tqdm(publications):
    publication_url = publication['href']  # URL for each publication
    pub_response = requests.get(publication_url)
    
    # print(pub_response.json())
    
    if pub_response.status_code == 200:
        pub_data = pub_response.json()
        # Process each publication (e.g., get the authors)
        # print(f"Title: {pub_data.get('naslov')}")
        title = pub_data.get('naslov')
        authors = []
        if 'osobeResources' in pub_data:
            superviser_flag = False
            for author in pub_data['osobeResources']['_embedded']['osobe']:
                if author['funkcija']['id'] in [905, 907]:
                    if author['funkcija']['id'] == 907:
                        superviser_flag = True
                    # print(f" - {author['ime']} {author['prezime']} {author['crorisId']}")
                    authors.append({"name": author['ime'], "surname": author['prezime'], "authorId": author['crorisId'], "role": author['funkcija']})
                    
        else:
            print("No authors found")
        # print("\n")
        if superviser_flag:
            authors.extend(pub_data.get('autori'))
        
        if len(authors) < 1:
            authors.append("no_authors")
            
        year = pub_data.get('godina')
        DOI = pub_data.get('crosbiId')
        
        paper_data = {
            "Title": title,
            "DOI": DOI,
            "Authors": authors
        }
        
        data_list.append(paper_data)
        
    else:
        print(f"Error fetching data from {publication_url}. Status code: {pub_response.status_code}")
        
df = pd.DataFrame(data_list)
        
        



 21%|██▏       | 222/1038 [00:17<01:01, 13.37it/s]

No authors found


 42%|████▏     | 435/1038 [00:35<00:46, 13.06it/s]

No authors found


 47%|████▋     | 483/1038 [00:38<00:43, 12.67it/s]

...

No authors found


 99%|█████████▊| 1024/1038 [01:26<00:01, 12.93it/s]

No authors found
No authors found


100%|██████████| 1038/1038 [01:27<00:00, 11.82it/s]

No authors found


In [52]:
print(df.info())
# Calculate the percentage of rows where the "Authors" column contains the list ["no_authors"]
percentage = len(df[df["Authors"].apply(lambda x: x == ["no_authors"])]) / len(df) * 100

# Print the percentage
print(percentage)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1038 entries, 0 to 1037
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Title    1038 non-null   object
 1   DOI      1038 non-null   int64 
 2   Authors  1038 non-null   object
dtypes: int64(1), object(2)
memory usage: 24.5+ KB
None
1.5414258188824663


In [53]:
print(df[df["Authors"].apply(lambda x: x == ["no_authors"])])

                                                  Title     DOI       Authors
0     2. Radionica znanstvenog programa "Napredne mr...    7710  [no_authors]
1     3. Radionica znanstvenog programa "Napredne mr...    7711  [no_authors]
2                                   Speech Technologies    8806  [no_authors]
3                      Speech and Language Technologies    8809  [no_authors]
4     4. Radionica znanstvenog programa "Napredne mr...   10385  [no_authors]
15    Znanstveni skup doktorskih studenata informati...   23191  [no_authors]
480           Automobile liability insurance data model  560582  [no_authors]
648   Wireless Network Security recommendations Usin...  624597  [no_authors]
881   Sentiment of the Tweets on Russo-Ukrainian War...  737813  [no_authors]
888                                   Uvod u FORTRAN 77  746873  [no_authors]
1005  Znanstveni skup doktorskih studenata informati...  814035  [no_authors]
1006  Znanstveni skup doktorskih studenata informati...  814035 

In [54]:
for author in df:
    print(author)

['no_authors']
['no_authors']
['no_authors']
['no_authors']
['no_authors']
[{'name': 'Mile', 'surname': 'Pavlić', 'authorId': 8822, 'role': {'id': 905, 'naziv': 'autor/i'}}]
[{'name': 'Mile', 'surname': 'Pavlić', 'authorId': 8822, 'role': {'id': 905, 'naziv': 'autor/i'}}, {'name': 'Velimir', 'surname': 'Srića', 'authorId': 22062, 'role': {'id': 905, 'naziv': 'autor/i'}}]
[{'name': 'Velimir', 'surname': 'Srića', 'authorId': 22062, 'role': {'id': 905, 'naziv': 'autor/i'}}, {'name': 'Mile', 'surname': 'Pavlić', 'authorId': 8822, 'role': {'id': 905, 'naziv': 'autor/i'}}]
[{'name': 'Mile', 'surname': 'Pavlić', 'authorId': 8822, 'role': {'id': 905, 'naziv': 'autor/i'}}]
[{'name': 'Mile', 'surname': 'Pavlić', 'authorId': 8822, 'role': {'id': 905, 'naziv': 'autor/i'}}]
[{'name': 'Mario', 'surname': 'Radovan', 'authorId': 22498, 'role': {'id': 905, 'naziv': 'autor/i'}}]
[{'name': 'Mario', 'surname': 'Radovan', 'authorId': 22498, 'role': {'id': 905, 'naziv': 'autor/i'}}]
[{'name': 'Mario', 'surn